In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# <span style="color:red"> ch6_스마트번역기(RNN기반의 Seq2Seq) </span>
- 인코더 LSTM과 디코더 LSTM을 붙여놓아 Seq2Seq방식(Sequence to Sequence)
- Google Neural Machine Translation(GNMT)
- LSTM은 RNN기반이다. 
- 자연어 생성 알고리즘
# 1. 패키지 import & 하이퍼파라미터
- 하이퍼파라미터: 모델의 정확도 및 학습속도에 영향을 미치는 변수

In [2]:
import pandas as pd
import numpy as np
from time import time

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM
from tensorflow.keras.utils import to_categorical

# 하이퍼 파라미터
MY_HIDDEN = 128
MY_EPOCH = 500

# 2. 번역데이터 불러오기

In [6]:
raw = pd.read_csv('data/translate.csv', header=None)
raw.values  # 어레이로 변경
eng_kor = raw.values.tolist() # 데이터프레임을 list로 변환
print('영어-한글 번역 데이터 : ', eng_kor[:3])
print('영어-한글 번역 데이터 수: ', len(eng_kor))

영어-한글 번역 데이터 :  [['cold', '감기'], ['come', '오다'], ['cook', '요리']]
영어-한글 번역 데이터 수:  110


# 3. 영어 알파벳과 한글문자 리스트 만들기 

In [33]:
e_alpha = [ c for c in 'SEPabcdefghijklmnopqrstuvwxyz']
#print(e_alpha)
#{ c:i for i, c in enumerate(e_alpha)}
# 한글 문자 리스트
korean = set(''.join([kor for eng, kor in eng_kor]))
korean = list(korean)
korean.sort()

k_alpha = pd.read_csv('data/korean.csv', header=None)[0].tolist()
print(k_alpha)

['가', '각', '간', '감', '개', '거', '것', '게', '계', '고', '관', '광', '구', '굴', '규', '그', '금', '기', '깊', '나', '날', '남', '내', '넓', '녀', '노', '놀', '농', '높', '뉴', '늦', '다', '단', '도', '동', '들', '람', '랑', '래', '램', '류', '름', '릎', '리', '많', '망', '매', '머', '먼', '멍', '메', '명', '모', '목', '무', '물', '미', '바', '반', '방', '번', '복', '부', '분', '붕', '비', '뿌', '사', '상', '색', '생', '서', '선', '소', '손', '수', '쉽', '스', '시', '식', '실', '싸', '아', '약', '얇', '어', '언', '얼', '여', '연', '오', '옥', '왼', '요', '용', '우', '운', '움', '위', '유', '은', '을', '음', '의', '이', '익', '인', '읽', '입', '자', '작', '장', '적', '제', '좋', '주', '지', '짜', '쪽', '찾', '책', '출', '칙', '크', '키', '탈', '택', '통', '파', '팔', '편', '피', '핑', '한', '합', '해', '행', '험', '회', '획', '휴', '흐']


In [35]:
alpha = e_alpha + k_alpha
print('영어와 한글 알파벳 :', alpha)
alpha_total_size = len(alpha)
print('전체 알파벳갯수(원핫인코딩할 사이즈) :', alpha_total_size)
print('한글 알파벳 갯수 :', len(k_alpha))

영어와 한글 알파벳 : ['S', 'E', 'P', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '가', '각', '간', '감', '개', '거', '것', '게', '계', '고', '관', '광', '구', '굴', '규', '그', '금', '기', '깊', '나', '날', '남', '내', '넓', '녀', '노', '놀', '농', '높', '뉴', '늦', '다', '단', '도', '동', '들', '람', '랑', '래', '램', '류', '름', '릎', '리', '많', '망', '매', '머', '먼', '멍', '메', '명', '모', '목', '무', '물', '미', '바', '반', '방', '번', '복', '부', '분', '붕', '비', '뿌', '사', '상', '색', '생', '서', '선', '소', '손', '수', '쉽', '스', '시', '식', '실', '싸', '아', '약', '얇', '어', '언', '얼', '여', '연', '오', '옥', '왼', '요', '용', '우', '운', '움', '위', '유', '은', '을', '음', '의', '이', '익', '인', '읽', '입', '자', '작', '장', '적', '제', '좋', '주', '지', '짜', '쪽', '찾', '책', '출', '칙', '크', '키', '탈', '택', '통', '파', '팔', '편', '피', '핑', '한', '합', '해', '행', '험', '회', '획', '휴', '흐']
전체 알파벳갯수(원핫인코딩할 사이즈) : 171
한글 알파벳 갯수 : 142


# 4. 문자당 num를 갖는 dict 만들기

In [37]:
char_to_num = { c:i for i, c in enumerate(alpha)}
print(char_to_num)

{'S': 0, 'E': 1, 'P': 2, 'a': 3, 'b': 4, 'c': 5, 'd': 6, 'e': 7, 'f': 8, 'g': 9, 'h': 10, 'i': 11, 'j': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'p': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'x': 26, 'y': 27, 'z': 28, '가': 29, '각': 30, '간': 31, '감': 32, '개': 33, '거': 34, '것': 35, '게': 36, '계': 37, '고': 38, '관': 39, '광': 40, '구': 41, '굴': 42, '규': 43, '그': 44, '금': 45, '기': 46, '깊': 47, '나': 48, '날': 49, '남': 50, '내': 51, '넓': 52, '녀': 53, '노': 54, '놀': 55, '농': 56, '높': 57, '뉴': 58, '늦': 59, '다': 60, '단': 61, '도': 62, '동': 63, '들': 64, '람': 65, '랑': 66, '래': 67, '램': 68, '류': 69, '름': 70, '릎': 71, '리': 72, '많': 73, '망': 74, '매': 75, '머': 76, '먼': 77, '멍': 78, '메': 79, '명': 80, '모': 81, '목': 82, '무': 83, '물': 84, '미': 85, '바': 86, '반': 87, '방': 88, '번': 89, '복': 90, '부': 91, '분': 92, '붕': 93, '비': 94, '뿌': 95, '사': 96, '상': 97, '색': 98, '생': 99, '서': 100, '선': 101, '소': 102, '손': 103, '수': 104, '쉽': 105, '스': 106, '시': 107, '식': 108, '실': 109, '싸': 110,

In [38]:
data = eng_kor[0]
print(data)
print(char_to_num['c'], char_to_num['o'], char_to_num['l'], char_to_num['d'])
print('인코더 입력(원핫인코딩전) :',[char_to_num[c] for c in data[0]])
print('디코더 입력(원핫인코딩전) :', [char_to_num[c] for c in 'S'+data[1]])
print('디코더 출력(원핫인코딩X) :', [char_to_num[c] for c in data[1]+'E'])

['cold', '감기']
5 17 14 6
인코더 입력(원핫인코딩전) : [5, 17, 14, 6]
디코더 입력(원핫인코딩전) : [0, 32, 46]
디코더 출력(원핫인코딩X) : [32, 46, 1]


In [47]:
# 희소 행렬의 원핫인코딩 방법 1 (희소행렬에서는 pd.get_dummies([2,9,7]) 안 씀)
# pd.get_dummies([5,17,14,6])
to_categorical([5, 7, 4, 6], num_classes=10) #alpha_total_size

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.]], dtype=float32)

In [51]:
# 희소 행렬의 원핫인코딩 방법 2 num_classes 와 사이즈가 같은 단위행렬을 만들어 각 행을 가져오면 된다
np.eye(10)[[5, 7, 4, 6]]

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.]])

# 5. 인코더 입력, 디코더 입력, 디코더 출력
- 인코더 입력 : 영어 알파벳 -> 숫자 -> 원핫인코딩(110, 4, 171)
- 디코더 입력 : 'S' + 한글문자 -> 숫자 -> 원핫인코딩(110, 3, 171)
- 디코더 출력(타겟) : 한글문자 + 'E' -> 숫자 shape (110, 3)인 리스트 -> np 배열로 변경 (110,3,1)

In [57]:
eng_kor[:3]

[['cold', '감기'], ['come', '오다'], ['cook', '요리']]

In [67]:
# 함수를 만들자. 학습도 학습이지만, 예측등을 위해서도 손쉽게 원핫인코딩을 할 수 있어야 한다.
def encoding(eng_kor = eng_kor):
    enc_in = []
    dec_in = []
    dec_out = []
    for data in eng_kor:
        # 인코더 입력데이터(영어 알파벳 -> 숫자 -> 원핫인코딩(110, 4, 171))
        eng = [ char_to_num[c] for c in data[0] ]
        eng_one = np.eye(alpha_total_size)[eng]
        # print('영어 :', eng, eng_one)
        enc_in.append(eng_one)   # eng_one의 shaep : 4,171
        
        # 디코더 입력데이터('S' + 한글 -> 숫자 -> 원핫인코딩(110, 3, 171)))
        kor = [ char_to_num[c] for c in 'S'+data[1] ]
        kor_one = to_categorical(kor, num_classes=alpha_total_size)   # kor_one의 사이즈 : 3, 171
        # == kor_one = np.eye(alpha_total_size)[kor]
        # print(kor, kor_one)
        dec_in.append(kor_one)
        
        # 디코더 출력데이터(한글 + 'E' -> 숫자)
        kor = [ char_to_num[c] for c in data[1]+'E']
        #print(kor) 
        dec_out.append(kor)
    return enc_in, dec_in, dec_out
encoding(eng_kor[0])

([array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]),
  array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0

In [69]:
sample = [['cold', '감기'], ['wood', '나무']]
x_enc, x_dec, y_dec = encoding(sample)

In [76]:
X_enc = np.array(x_enc)
X_dec = np.array(x_dec)
Y_dec = np.array(y_dec)

In [77]:
type(x_enc)  # 리스트 인것이라 numpy 배열로 바꿔줘야 한다

list

In [81]:
type(X_enc), X_enc.shape, X_dec.shape, Y_dec.shape # Y.dec -> 축을 증가시켜야 학습 가능 (2,3,1)

(numpy.ndarray, (2, 4, 171), (2, 3, 171), (2, 3))

In [82]:
# 축 증가 1
Y_dec.reshape(2,3,1)

array([[[32],
        [46],
        [ 1]],

       [[48],
        [83],
        [ 1]]])

In [83]:
# 축 증가 방법 2 : 맨 마지막 축 증가하는 함수
np.expand_dims(Y_dec, axis=-1)  

array([[[32],
        [46],
        [ 1]],

       [[48],
        [83],
        [ 1]]])

In [84]:
# 축 증가 방법 3 : 이미지 딥러닝할때 많이 사용함
Y_dec[..., np.newaxis]

array([[[32],
        [46],
        [ 1]],

       [[48],
        [83],
        [ 1]]])

In [88]:
# 축 증가 방법 4
Y_dec[:,:,None].shape

(2, 3, 1)

# 6. 전체 입력데이터, 타겟데이터 준비

In [90]:
x_enc, x_dec, y_dec = encoding(eng_kor)
X_enc = np.array(x_enc)
X_dec = np.array(x_dec)
Y_dec = np.expand_dims(y_dec, axis=-1)  

X_enc.shape, X_dec.shape, Y_dec.shape

((110, 4, 171), (110, 3, 171), (110, 3, 1))

# 7. 모델 구현

In [96]:
# 인코더 LSTM

ENC_IN = Input(shape=(4, alpha_total_size)) # 인풋층 생성 / alpha_total_size : 171
LSTM(units=MY_HIDDEN, # MY_HIDDEN : 128
        return_state=True  # return_state=True  h 값과 c값 받기
        # return_sequences=False # return_sequences=False LSTM 윗 출력 안받음
        )(ENC_IN)  

[<KerasTensor: shape=(None, 128) dtype=float32 (created by layer 'lstm_5')>,
 <KerasTensor: shape=(None, 128) dtype=float32 (created by layer 'lstm_5')>,
 <KerasTensor: shape=(None, 128) dtype=float32 (created by layer 'lstm_5')>]

In [97]:
# 첫번째는 위로(Dense) 올라가는 값이지만, 인코더 파트에는 넣지 않을 예정이라 더미로 받아버린다
_, state_h, state_c = LSTM(units=MY_HIDDEN, # MY_HIDDEN : 128
                            return_state=True)(ENC_IN)  

In [98]:
# 인코더와 디코더 연결고리
LINK = [state_h, state_c]

# 디코더 LSTM
DEC_IN = Input(shape=(3, alpha_total_size))
DEC_MID = LSTM(units=MY_HIDDEN,
              # return_state=False, 기본값
              return_sequences=True # 윗 출력 받음
              )(DEC_IN,
               initial_state=LINK)

# 최종 출력층
DEC_OUT = Dense(units=alpha_total_size,
               activation='softmax')(DEC_MID)

In [99]:
# 모델
model = Model(inputs=[ENC_IN, DEC_IN],
             outputs=DEC_OUT)
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_6 (InputLayer)           [(None, 4, 171)]     0           []                               
                                                                                                  
 input_7 (InputLayer)           [(None, 3, 171)]     0           []                               
                                                                                                  
 lstm_6 (LSTM)                  [(None, 128),        153600      ['input_6[0][0]']                
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                                                                              

# 8. 학습과정 설정 & 학습

In [100]:
# 학습 과정 설정
model.compile(loss='sparse_categorical_crossentropy',
             optimizer = 'rmsprop',
             metrics=['accuracy'])  # metrics 생략해도 괜찮다. loss만 로그 출력된다
begin = time()
model.fit([X_enc, X_dec], Y_dec,
         batch_size=1000,
         epochs=MY_EPOCH,
         verbose=1)
end = time()
print(f'학습시간 : {end-begin}초')

Epoch 1/500
1/1 [==============================] - 3s 3s/step - loss: 5.1435 - accuracy: 0.0061
Epoch 2/500
1/1 [==============================] - 0s 25ms/step - loss: 5.1125 - accuracy: 0.2879
Epoch 3/500
1/1 [==============================] - 0s 23ms/step - loss: 5.0834 - accuracy: 0.3333
Epoch 4/500
1/1 [==============================] - 0s 25ms/step - loss: 5.0456 - accuracy: 0.3333
Epoch 5/500
1/1 [==============================] - 0s 21ms/step - loss: 4.9897 - accuracy: 0.3333
Epoch 6/500
1/1 [==============================] - 0s 19ms/step - loss: 4.9001 - accuracy: 0.3333
Epoch 7/500
1/1 [==============================] - 0s 20ms/step - loss: 4.7483 - accuracy: 0.3333
Epoch 8/500
1/1 [==============================] - 0s 18ms/step - loss: 4.4826 - accuracy: 0.3333
Epoch 9/500
1/1 [==============================] - 0s 20ms/step - loss: 4.0398 - accuracy: 0.3333
Epoch 10/500
1/1 [==============================] - 0s 19ms/step - loss: 3.5527 - accuracy: 0.3333
Epoch 11/500
1/1 [===

1/1 [==============================] - 0s 18ms/step - loss: 2.7628 - accuracy: 0.3848
Epoch 84/500
1/1 [==============================] - 0s 19ms/step - loss: 2.7513 - accuracy: 0.3879
Epoch 85/500
1/1 [==============================] - 0s 18ms/step - loss: 2.7465 - accuracy: 0.3576
Epoch 86/500
1/1 [==============================] - 0s 18ms/step - loss: 2.7760 - accuracy: 0.4030
Epoch 87/500
1/1 [==============================] - 0s 17ms/step - loss: 2.8578 - accuracy: 0.3394
Epoch 88/500
1/1 [==============================] - 0s 18ms/step - loss: 2.7764 - accuracy: 0.4091
Epoch 89/500
1/1 [==============================] - 0s 17ms/step - loss: 2.7201 - accuracy: 0.3545
Epoch 90/500
1/1 [==============================] - 0s 17ms/step - loss: 2.6900 - accuracy: 0.3909
Epoch 91/500
1/1 [==============================] - 0s 16ms/step - loss: 2.6767 - accuracy: 0.3879
Epoch 92/500
1/1 [==============================] - 0s 18ms/step - loss: 2.6644 - accuracy: 0.3909
Epoch 93/500
1/1 [=====

1/1 [==============================] - 0s 17ms/step - loss: 1.4307 - accuracy: 0.7939
Epoch 166/500
1/1 [==============================] - 0s 17ms/step - loss: 1.4158 - accuracy: 0.8091
Epoch 167/500
1/1 [==============================] - 0s 16ms/step - loss: 1.4010 - accuracy: 0.8182
Epoch 168/500
1/1 [==============================] - 0s 17ms/step - loss: 1.3863 - accuracy: 0.8182
Epoch 169/500
1/1 [==============================] - 0s 18ms/step - loss: 1.3715 - accuracy: 0.8273
Epoch 170/500
1/1 [==============================] - 0s 17ms/step - loss: 1.3567 - accuracy: 0.8333
Epoch 171/500
1/1 [==============================] - 0s 17ms/step - loss: 1.3417 - accuracy: 0.8394
Epoch 172/500
1/1 [==============================] - 0s 17ms/step - loss: 1.3267 - accuracy: 0.8394
Epoch 173/500
1/1 [==============================] - 0s 17ms/step - loss: 1.3115 - accuracy: 0.8364
Epoch 174/500
1/1 [==============================] - 0s 17ms/step - loss: 1.2961 - accuracy: 0.8424
Epoch 175/500


1/1 [==============================] - 0s 19ms/step - loss: 0.4272 - accuracy: 0.9788
Epoch 248/500
1/1 [==============================] - 0s 16ms/step - loss: 0.4195 - accuracy: 0.9818
Epoch 249/500
1/1 [==============================] - 0s 16ms/step - loss: 0.4118 - accuracy: 0.9818
Epoch 250/500
1/1 [==============================] - 0s 16ms/step - loss: 0.4041 - accuracy: 0.9909
Epoch 251/500
1/1 [==============================] - 0s 17ms/step - loss: 0.3964 - accuracy: 0.9909
Epoch 252/500
1/1 [==============================] - 0s 17ms/step - loss: 0.3886 - accuracy: 0.9909
Epoch 253/500
1/1 [==============================] - 0s 17ms/step - loss: 0.3809 - accuracy: 0.9909
Epoch 254/500
1/1 [==============================] - 0s 18ms/step - loss: 0.3732 - accuracy: 0.9909
Epoch 255/500
1/1 [==============================] - 0s 17ms/step - loss: 0.3654 - accuracy: 0.9909
Epoch 256/500
1/1 [==============================] - 0s 17ms/step - loss: 0.3577 - accuracy: 0.9909
Epoch 257/500


1/1 [==============================] - 0s 17ms/step - loss: 0.0675 - accuracy: 1.0000
Epoch 330/500
1/1 [==============================] - 0s 34ms/step - loss: 0.0660 - accuracy: 1.0000
Epoch 331/500
1/1 [==============================] - 0s 21ms/step - loss: 0.0644 - accuracy: 1.0000
Epoch 332/500
1/1 [==============================] - 0s 21ms/step - loss: 0.0629 - accuracy: 1.0000
Epoch 333/500
1/1 [==============================] - 0s 16ms/step - loss: 0.0614 - accuracy: 1.0000
Epoch 334/500
1/1 [==============================] - 0s 20ms/step - loss: 0.0600 - accuracy: 1.0000
Epoch 335/500
1/1 [==============================] - 0s 18ms/step - loss: 0.0585 - accuracy: 1.0000
Epoch 336/500
1/1 [==============================] - 0s 21ms/step - loss: 0.0570 - accuracy: 1.0000
Epoch 337/500
1/1 [==============================] - 0s 20ms/step - loss: 0.0556 - accuracy: 1.0000
Epoch 338/500
1/1 [==============================] - 0s 19ms/step - loss: 0.0542 - accuracy: 1.0000
Epoch 339/500


1/1 [==============================] - 0s 15ms/step - loss: 0.0074 - accuracy: 1.0000
Epoch 412/500
1/1 [==============================] - 0s 16ms/step - loss: 0.0073 - accuracy: 1.0000
Epoch 413/500
1/1 [==============================] - 0s 16ms/step - loss: 0.0075 - accuracy: 1.0000
Epoch 414/500
1/1 [==============================] - 0s 16ms/step - loss: 0.0078 - accuracy: 1.0000
Epoch 415/500
1/1 [==============================] - 0s 16ms/step - loss: 0.0079 - accuracy: 1.0000
Epoch 416/500
1/1 [==============================] - 0s 18ms/step - loss: 0.0076 - accuracy: 1.0000
Epoch 417/500
1/1 [==============================] - 0s 18ms/step - loss: 0.0068 - accuracy: 1.0000
Epoch 418/500
1/1 [==============================] - 0s 19ms/step - loss: 0.0061 - accuracy: 1.0000
Epoch 419/500
1/1 [==============================] - 0s 20ms/step - loss: 0.0057 - accuracy: 1.0000
Epoch 420/500
1/1 [==============================] - 0s 19ms/step - loss: 0.0055 - accuracy: 1.0000
Epoch 421/500


1/1 [==============================] - 0s 16ms/step - loss: 9.6507e-04 - accuracy: 1.0000
Epoch 494/500
1/1 [==============================] - 0s 17ms/step - loss: 9.4118e-04 - accuracy: 1.0000
Epoch 495/500
1/1 [==============================] - 0s 17ms/step - loss: 9.1769e-04 - accuracy: 1.0000
Epoch 496/500
1/1 [==============================] - 0s 17ms/step - loss: 8.9458e-04 - accuracy: 1.0000
Epoch 497/500
1/1 [==============================] - 0s 17ms/step - loss: 8.7186e-04 - accuracy: 1.0000
Epoch 498/500
1/1 [==============================] - 0s 17ms/step - loss: 8.4956e-04 - accuracy: 1.0000
Epoch 499/500
1/1 [==============================] - 0s 19ms/step - loss: 8.2764e-04 - accuracy: 1.0000
Epoch 500/500
1/1 [==============================] - 0s 21ms/step - loss: 8.0612e-04 - accuracy: 1.0000
학습시간 : 13.65190315246582초


In [101]:
model.evaluate([X_enc, X_dec], Y_dec)

4/4 [==============================] - 3s 11ms/step - loss: 7.8501e-04 - accuracy: 1.0000


[0.0007850064430385828, 1.0]

# 9. 모델사용

In [103]:
easy_test = [['cold', 'PP'],
            ['fact', 'PP'],
            ['love', 'PP'],
            ['luck', 'PP'],
            ['milk', 'PP']]
enc_in, dec_in, _ = encoding(easy_test)
enc_in = np.array(enc_in)
dec_in = np.array(dec_in)
enc_in.shape, dec_in.shape

((5, 4, 171), (5, 3, 171))

In [106]:
# 위의 문제 예측하기
pred = model.predict([enc_in, dec_in])
pred.argmax(axis=-1)

1/1 [==============================] - 0s 24ms/step


array([[ 32,  46,   1],
       [ 96, 109,   1],
       [ 96,  66,   1],
       [165, 125,   1],
       [124, 128,   1]], dtype=int64)

In [118]:
# cold => 감기

# num_to_char = { n:c for c, n in char_to_num}
# alpha[32]

for test, yhat in zip(easy_test, pred):
    # print(test[0], yhat.argmax(axis=-1)[0])
    eng = test[0]
    hat = np.argmax(yhat, axis=-1)
    kor = ''.join([alpha[h] for h in hat[:-1]])
    print(f'{eng} => {kor}')
    

cold => 감기
fact => 사실
love => 사랑
luck => 행운
milk => 우유


In [110]:
type(char_to_num)

dict

In [138]:
# 어려운 문제
hard_test = [['lvoe', 'PP'],
            ['love', 'PP'],
            ['loev', 'PP'],
            ['ruck', 'PP'],
            ['milf', 'PP']]
enc_in, dec_in, dec_out = encoding(hard_test)
enc_in = np.array(enc_in)
dec_in = np.array(dec_in)
pred = model.predict([enc_in, dec_in]).argmax(axis=-1)

[ ''.join([alpha[h] for h in hat[:-1]]) for hat in pred ]

1/1 [==============================] - 0s 23ms/step


['사랑', '사랑', '사랑', '행위', '우유']

In [137]:
for test, yhat in zip(hard_test, pred):
    eng = test[0]
    kor = ''.join(alpha[h] for h in yhat[:-1])
    print(f'{eng} => {kor}')

lvoe => 사랑
love => 사랑
loev => 사랑
olve => 사랑
milf => 우유


In [129]:
hat

0

In [136]:
for test, yhat in zip(hard_test, pred):
    print(test[0], yhat)

lvoe [96 66  1]
love [96 66  1]
loev [96 66  1]
olve [96 66  1]
milf [124 128   1]
